# Transitions of Properties for one Status to another

In [64]:
import pandas as pd
import re
from pathlib import Path
from IPython.display import display, Markdown
import numpy as np
from IPython.display import display, HTML

In [36]:
# Helpers for generating transition tables

def print_result_only_complete_triplets(df, image_col="image"):
    # --------------------------
    # numeric safety
    # --------------------------
    df["eps"] = pd.to_numeric(df["eps"], errors="coerce")
    df["k"]   = pd.to_numeric(df["k"], errors="coerce")
    df["lb_minus_rhs"] = pd.to_numeric(df["lb_minus_rhs"], errors="coerce")

    # --------------------------
    # make sure domains_visited exists
    # --------------------------
    assert "domains_visited" in df.columns

    # --------------------------
    # BnB flag
    # --------------------------
    df["bnb_entered"] = pd.to_numeric(df["domains_visited"], errors="coerce").fillna(0) > 0

    # --------------------------
    # VARIANT from tag text
    # (IMPORTANT: check nonmask BEFORE mask because "nonmask" contains "mask")
    # --------------------------
    tag = df["tag"].astype(str).str.lower()

    df["variant"] = np.select(
        [
            tag.str.contains("global"),
            tag.str.contains("fixnonmask") | tag.str.contains("object"),
            tag.str.contains("fixmask") | tag.str.contains("background"),
        ],
        ["Global", "Object", "Background"],
        default="Unknown",
    )

    # optionally keep only these 3 variants (helps avoid weird rows)
    df = df[df["variant"].isin(["Global", "Object", "Background"])].copy()

    # --------------------------
    # result mapping
    # --------------------------
    RESULT_RENAME = {
        "sat True": "Unsafe (counterexample)",
        "unsat False": "Safe (proved)",
        "timeout False": "Unknown",
        "error": "Unknown",
        "sat False": "Unsafe (counterexample during initial bounding)",
    }

    df["result_label"] = df["result"].astype(str).map(RESULT_RENAME).fillna(df["result"].astype(str))

    # --------------------------
    # lower bound status
    # --------------------------
    df["glb_status"] = np.where(
        np.isfinite(df["lb_minus_rhs"]),
        "Finite lower bound",
        "Lower bound = -inf",
    )

    # --------------------------
    # SUMMARY (includes variant)
    # --------------------------
    summary = (
        df.groupby(["variant", "result_label", "bnb_entered", "glb_status"])
          .size()
          .reset_index(name="Count")
          .sort_values(["variant", "Count"], ascending=[True, False])
    )

    return summary


def pivot_summary_with_totals(summary):
    s = summary.copy()

    # 1) One "Type" column
    s["Type"] = (
        s["result_label"]
        + " | bnb=" + s["bnb_entered"].astype(str)
        + " | " + s["glb_status"]
    )

    # 2) Pivot
    table = (
        s.pivot_table(
            index="Type",
            columns="variant",
            values="Count",
            aggfunc="sum",
            fill_value=0,
        )
        .reset_index()
    )

    table.columns.name = None

    # 3) Ensure consistent column order
    for c in ["Global", "Object", "Background"]:
        if c not in table.columns:
            table[c] = 0
    table = table[["Type", "Global", "Object", "Background"]]

    # 4) Add Total column
    table["Total"] = table[["Global", "Object", "Background"]].sum(axis=1)

    # 5) Sort by Total
    table = table.sort_values("Total", ascending=False).reset_index(drop=True)

    # 6) Add final totals row (requested)
    totals_row = pd.DataFrame([{
        "Type": "TOTAL (all types)",
        "Global": int(table["Global"].sum()),
        "Object": int(table["Object"].sum()),
        "Background": int(table["Background"].sum()),
        "Total": int(table["Total"].sum()),
    }])

    table = pd.concat([table, totals_row], ignore_index=True)

    pd.set_option("display.max_colwidth", None)
    pd.set_option("display.width", 2000)
    pd.set_option("display.max_columns", None)
    pd.set_option("display.expand_frame_repr", False)

    return table

In [37]:
# To display result for a csv file

CSV_PATH = f"results/exp_1/triplets_k50176.0_eps0.0001_239_imgs.csv"
df = pd.read_csv(CSV_PATH)
summary = print_result_only_complete_triplets(df)
table = pivot_summary_with_totals(summary)
table

,Type,Global,Object,Background,Total
0,Unsafe (counterexample) | bnb=False | Lower bound = -inf,146,146,127,419
1,Safe (proved) | bnb=False | Finite lower bound,16,60,95,171
2,Unknown | bnb=True | Finite lower bound,59,32,0,91
3,Unknown | bnb=False | Lower bound = -inf,18,1,0,19
4,Unsafe (counterexample during initial bounding) | bnb=False | Lower bound = -inf,0,0,17,17
5,TOTAL (all types),239,239,239,717


In [86]:
# To display results for all csv files in a folder in experiment 1

FNAME_RE = re.compile(r"triplets_k(?P<k>[-+]?\d*\.?\d+)_eps(?P<eps>[-+]?\d*\.?\d+)", re.I)

def parse_k_eps(p):
    m = FNAME_RE.search(p.name)
    return (float(m["k"]), float(m["eps"])) if m else (None, None)

def build_big(folder):
    parts = []
    for fp in sorted(Path(folder).glob("triplets_*.csv")):
        k, eps = parse_k_eps(fp)
        d = pd.read_csv(fp)

        # inject if missing
        if "k" not in d: d["k"] = k
        if "eps" not in d: d["eps"] = eps

        # ---- your logic (compact) ----
        d["eps"] = pd.to_numeric(d["eps"], errors="coerce")
        d["k"]   = pd.to_numeric(d["k"], errors="coerce")
        d["lb_minus_rhs"] = pd.to_numeric(d.get("lb_minus_rhs", np.nan), errors="coerce")

        d["bnb_entered"] = pd.to_numeric(d["domains_visited"], errors="coerce").fillna(0) > 0

        tag = d["tag"].astype(str).str.lower()
        d["variant"] = np.select(
            [tag.str.contains("global"),
             tag.str.contains("fixnonmask") | tag.str.contains("object"),
             tag.str.contains("fixmask") | tag.str.contains("background")],
            ["G", "O", "B"],  # SHORT labels
            default="X",
        )
        d = d[d["variant"].isin(["G","O","B"])].copy()

        RESULT_RENAME = {
            "sat True": "Unsafe",
            "unsat False": "Safe",
            "timeout False": "Unknown",
            "error": "Unknown",
            "sat False": "InitUnsafe",
        }
        d["result_label"] = d["result"].astype(str).map(RESULT_RENAME).fillna(d["result"].astype(str))
        d["glb_status"] = np.where(np.isfinite(d["lb_minus_rhs"]), "fin", "-inf")

        s = (d.groupby(["variant","result_label","bnb_entered","glb_status"])
               .size().reset_index(name="Count"))
        s["Type"] = (
            s["result_label"]
            + "|bnb=" + s["bnb_entered"].astype(int).astype(str)
            + "|" + s["glb_status"]
        )

        t = (s.pivot_table(index="Type", columns="variant", values="Count", aggfunc="sum", fill_value=0)
               .reindex(columns=["G","O","B"], fill_value=0))
        t["T"] = t.sum(axis=1)

        # totals row
        tot = pd.DataFrame([t.sum(axis=0)], index=["TOTAL"])
        t = pd.concat([t, tot], axis=0)

        # MultiIndex columns (k, eps, metric)
        t.columns = pd.MultiIndex.from_product([[k],[eps], t.columns])
        parts.append(t)

    big = pd.concat(parts, axis=1).fillna(0).astype(int)

    # reorder metrics inside each (k,eps): G,O,B,T
    new_cols = []
    for k in sorted(set(big.columns.get_level_values(0))):
        for eps in sorted(set(big.loc[:, k].columns.get_level_values(0))):
            for m in ["G","O","B","T"]:
                if (k, eps, m) in big.columns:
                    new_cols.append((k, eps, m))
    big = big.loc[:, new_cols]

    # move TOTAL row to bottom (in case)
    mask = big.index.astype(str).str.contains("TOTAL", case=False, na=False)
    big = pd.concat([big.loc[~mask], big.loc[mask]])

    return big

def tiny_style(big):
    # --- helper for nice float labels (no trailing zeros / no scientific noise) ---
    def fmt_num(x, max_dec=6):
        try:
            x = float(x)
        except Exception:
            return str(x)
        # fixed decimals then strip trailing zeros + dot
        s = f"{x:.{max_dec}f}".rstrip("0").rstrip(".")
        # keep at least "0" if it becomes empty
        return s if s else "0"

    # --- build display column labels like: k=10 | eps=0.001 (instead of 10.000000, 0.001000) ---
    col0 = [f"k={fmt_num(k)}" for k in big.columns.get_level_values(0)]
    col1 = [f"eps={fmt_num(eps)}" for eps in big.columns.get_level_values(1)]
    col2 = list(big.columns.get_level_values(2))

    big_disp = big.copy()
    big_disp.columns = pd.MultiIndex.from_arrays([col0, col1, col2])

    # --- find column positions where an eps-block ends (we end blocks at metric 'T') ---
    # your metrics order is G,O,B,T; so a clean "divider after each epsilon" = after each 'T'
    eps_vals = list(big.columns.get_level_values(1))
    metrics  = list(big.columns.get_level_values(2))
    divider_cols = [i for i, (eps, m) in enumerate(zip(eps_vals, metrics)) if m == "T"]

    # --- base style ---
    sty = (big_disp.style
        .set_properties(**{
            "font-size": "12px",
            "padding": "2px 6px",
            "text-align": "center",
            "white-space": "nowrap",
            "line-height": "1.05",
        })
        .set_table_styles([
            {"selector":"th", "props":[("font-size","12px"), ("padding","2px 6px"), ("white-space","nowrap")]},
            {"selector":"td", "props":[("font-size","12px"), ("padding","2px 6px"), ("white-space","nowrap")]},
            {"selector":"table", "props":[("border-collapse","collapse")]},
        ])
    )

    # --- add a vertical line after each eps block (after T) ---
    if divider_cols:
        # include both header + body cells
        col_props = [{"selector": f"td.col{i}", "props": [("border-right", "2px solid #888")]} for i in divider_cols]
        col_props += [{"selector": f"th.col{i}", "props": [("border-right", "2px solid #888")]} for i in divider_cols]
        sty = sty.set_table_styles(col_props, overwrite=False)

    return sty
# ---- RUN ----
FOLDER = "results/exp_1_only_global_common"   
big = build_big(FOLDER)
display(tiny_style(big))

# save flat csv with short headers
flat = big.copy()
flat.columns = [f"k{float(k):g}_e{float(eps):g}_{m}" for (k,eps,m) in flat.columns]
# flat.to_csv("all_k_eps_tiny_flat.csv", index=True)
# print("Saved: all_k_eps_tiny_flat.csv")


# ---- RUN ----
FOLDER = "results/exp_2_last_3_common"   
big = build_big(FOLDER)
display(tiny_style(big))

# save flat csv with short headers
flat = big.copy()
flat.columns = [f"k{float(k):g}_e{float(eps):g}_{m}" for (k,eps,m) in flat.columns]
# flat.to_csv("all_k_eps_tiny_flat.csv", index=True)
# print("Saved: all_k_eps_tiny_flat.csv")




In [84]:
import numpy as np
import pandas as pd

VARIANTS = ["Global", "Object", "Background"]

# EXACT labels you asked for (order in the matrix)
EXP_LABELS = [
    "Unsafe",
    "Unsafe (II)",          # initial bounding unsafe
    "Unknown (BnB)",
    "Unknown (no BnB)",
    "Safe",
    "Safe (BnB)",
]

def prepare_df(df: pd.DataFrame) -> pd.DataFrame:
    d = df.copy()

    # numeric safety
    for c in ["eps", "k", "lb_minus_rhs", "domains_visited"]:
        if c in d.columns:
            d[c] = pd.to_numeric(d[c], errors="coerce")

    assert "tag" in d.columns
    assert "result" in d.columns
    assert "domains_visited" in d.columns

    # BnB flag
    d["bnb_entered"] = pd.to_numeric(d["domains_visited"], errors="coerce").fillna(0) > 0

    # image id
    def extract_image(s: pd.Series) -> pd.Series:
        return s.astype(str).str.extract(r"(n\d+_[^_]+)", expand=False)

    img = extract_image(d["tag"])
    if "vnnlib" in d.columns:
        img = img.fillna(extract_image(d["vnnlib"]))
    d["image"] = img.fillna("UnknownImage")

    # variant from tag (nonmask BEFORE mask)
    tag = d["tag"].astype(str).str.lower()
    d["variant"] = np.select(
        [
            tag.str.contains("global"),
            tag.str.contains("fixnonmask") | tag.str.contains("object"),
            tag.str.contains("fixmask") | tag.str.contains("background"),
        ],
        ["Global", "Object", "Background"],
        default="Unknown",
    )
    d = d[d["variant"].isin(VARIANTS)].copy()

    return d


def expanded_label_from_raw_result(result: str, bnb: bool) -> str:
    """
    Map your raw results to the 6 labels.

    raw result values you told me exist:
      - 'sat True'      => Unsafe
      - 'sat False'     => Unsafe (II)  (initial bounding)
      - 'unsat False'   => Safe OR Safe (BnB)
      - 'timeout False' => Unknown (BnB) OR Unknown (no BnB)
      - 'error'         => Unknown (BnB) OR Unknown (no BnB)
    """
    r = str(result).strip()

    if r == "sat True":
        return "Unsafe"

    if r == "sat False":
        return "Unsafe (II)"

    if r == "unsat False":
        return "Safe (BnB)" if bnb else "Safe"

    if r in ["timeout False", "error"]:
        return "Unknown (BnB)" if bnb else "Unknown (no BnB)"

    # fallback (shouldn't happen)
    return "Unknown (BnB)" if bnb else "Unknown (no BnB)"


def build_wide_triplets(df: pd.DataFrame) -> pd.DataFrame:
    d = prepare_df(df)
    keys = ["image", "eps", "k"]

    # build an expanded label per row
    d["status_X"] = [
        expanded_label_from_raw_result(r, b)
        for r, b in zip(d["result"].astype(str), d["bnb_entered"].astype(bool))
    ]

    # pivot to one row per (image,eps,k) with 3 variant columns
    wide = (
        d.pivot_table(index=keys, columns="variant", values="status_X", aggfunc="first")
         .reset_index()
    )
    wide.columns.name = None

    # ensure columns exist
    for v in VARIANTS:
        if v not in wide.columns:
            wide[v] = np.nan

    # keep only complete triplets
    wide = wide.dropna(subset=VARIANTS).copy()

    # rename for matrix usage
    wide = wide.rename(columns={
        "Global": "Global_X",
        "Object": "Object_X",
        "Background": "Background_X",
    })

    return wide


def transition_matrix(wide: pd.DataFrame, from_col: str, to_col: str, to_name: str) -> pd.DataFrame:
    mat = pd.crosstab(wide[from_col], wide[to_col], dropna=False)

    # force EXACT label ordering + include missing labels as 0 rows/cols
    mat = mat.reindex(index=EXP_LABELS, columns=EXP_LABELS, fill_value=0).astype(int)

    mat["RowTotal"] = mat.sum(axis=1)
    mat.loc["ColTotal"] = mat.sum(axis=0)

    mat.index.name = from_col
    mat.columns.name = to_name
    return mat


# =========================
# RUN
# =========================
wide = build_wide_triplets(df)
m_go = transition_matrix(wide, "Global_X", "Object_X", "Object_X")
m_gb = transition_matrix(wide, "Global_X", "Background_X", "Background_X")




def style_matrix(mat: pd.DataFrame, title: str):
    df = mat.copy()

    # ensure ints display nicely
    for c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0).astype(int)

    # identify total row/col
    total_row = (df.index.astype(str) == "ColTotal")
    total_col = (df.columns.astype(str) == "RowTotal")

    sty = (
        df.style
        .set_caption(title)
        .format("{:d}")
        .set_properties(**{"text-align": "center"})
        .set_table_styles([
            {"selector": "caption",
             "props": [("caption-side", "top"),
                       ("font-size", "16px"),
                       ("font-weight", "700"),
                       ("text-align", "left"),
                       ("padding", "6px 0px")]},
            {"selector": "th", "props": [("text-align", "center")]},
            {"selector": "td", "props": [("text-align", "center")]},
        ])
        # subtle heatmap (excluding totals)
        .background_gradient(axis=None, subset=pd.IndexSlice[~total_row, df.columns[~total_col]])
        # bold totals row/col
        .set_properties(subset=pd.IndexSlice[total_row, :], **{"font-weight": "700"})
        .set_properties(subset=pd.IndexSlice[:, total_col], **{"font-weight": "700"})
    )
    return sty

# side-by-side display
left  = style_matrix(m_go, "GLOBAL → OBJECT (expanded labels)")
right = style_matrix(m_gb, "GLOBAL → BACKGROUND (expanded labels)")

display(HTML(
    "<div style='display:flex; gap:24px; align-items:flex-start;'>"
    f"<div>{left.to_html()}</div>"
    f"<div>{right.to_html()}</div>"
    "</div>"
))

Object_X,Unsafe,Unsafe (II),Unknown (BnB),Unknown (no BnB),Safe,Safe (BnB),RowTotal
Global_X,,,,,,,
Unsafe,146,0,0,0,0,0,146
Unsafe (II),0,0,0,0,0,0,0
Unknown (BnB),0,0,23,1,35,0,59
Unknown (no BnB),0,0,9,0,9,0,18
Safe,0,0,0,0,16,0,16
Safe (BnB),0,0,0,0,0,0,0
ColTotal,146,0,32,1,60,0,239
Background_X,Unsafe,Unsafe (II),Unknown (BnB),Unknown (no BnB),Safe,Safe (BnB),RowTotal
Global_X,,,,,,,
